# tiny-urdu: compare the 1k and 5k models (WER)
Each model speaks the same unseen UrduTTS sentences. Whisper (speech-to-text) writes down what it hears, and we count the wrong words.
**Lower is better.** The real recordings give the ceiling: even perfect speech is not scored 0 by Whisper.

Setup on Kaggle:
1. **New Dataset** -> upload `models/1k` and `models/5k` (each with `model.onnx` and `model.json`) -> name it `tiny-urdu-models`.
2. In this notebook: **Add Input** -> that dataset. Turn **Internet ON** and pick a **GPU**.
3. Run all cells.

In [ ]:
!pip install -q "huggingface-hub>=0.34,<1" "transformers>=4.40,<5" "datasets<4" jiwer onnxruntime librosa soundfile 2>&1 | grep -v -i -E 'dependency resolver|requires|^$' || true
!mkdir -p urdu

In [ ]:
N = 150        # test sentences
SKIP = 200     # skip the first test rows: some were used for validation during training
WHISPER = 'openai/whisper-large-v3'

## Urdu front end (same file as training)

In [ ]:
%%writefile urdu/frontend.py
"""Urdu front end for tiny-voca.

tiny-voca's 76 English phoneme symbols are kept (so the English checkpoint can
be fine-tuned) and NATIVE Urdu phones are appended: retroflex ʈ ɖ ɽ, aspirated
kʰ tʰ bʰ ..., q x ɣ ʋ, long/nasal vowels aː eː ẽ ã ... Every symbol is a label
of the wav2vec2 forced aligner, spelled exactly as in its vocab.json (nasal
vowels decomposed: base + U+0303). Mapping Urdu onto English symbols instead
(kʰ -> k h, ʈ -> t) made the aligner reject 998/1000 clips.

Input to ``phonemize``:
* ``"ipa:<phonemes>"``  the UrduTTS ``phonemes`` column (TRAINING)
* anything else         Roman Urdu, e.g. ``"aap kaise hain?"`` (INFERENCE)
  Capital T, D, R mark retroflex ʈ ɖ ɽ (``baRa`` = big, ``bura`` = bad).

Both routes go through ``_ipa_word``, so they produce one token language.
Stress marks are dropped (Roman Urdu has none to reproduce).
"""

import hashlib
import json
import re
import unicodedata

IPA_PREFIX = "ipa:"
VERSION = "urdu-2"
N = "̃"  # combining tilde (nasal)

# Appended to tiny-voca's SYMBOLS. APPEND-ONLY: reordering breaks checkpoints.
URDU_CONSONANTS = ("ʈ", "ɖ", "ɽ", "q", "x", "ɣ", "ʋ", "r", "ɟ", "c", "ʂ", "ʐ", "ɳ", "ɲ", "ɭ",
                   "kʰ", "pʰ", "tʰ", "ʈʰ", "bʰ", "dʰ", "ɖʰ", "ɡʰ", "ɟʰ", "cʰ")
URDU_VOWELS = ("a", "aː", "e", "eː", "o", "u", "ɛː", "ɪː", "ʊː",
               "a" + N, "e" + N, "e" + N + "ː", "o" + N, "i" + N, "u" + N, "ɛ" + N, "ɔ" + N, "ɐ" + N, "ɑ" + N)
URDU_PHONES = URDU_CONSONANTS + URDU_VOWELS
URDU_VOICELESS = ("ʈ", "q", "x", "c", "ʂ", "kʰ", "pʰ", "tʰ", "ʈʰ", "cʰ")

_IGNORED = set("ˈˌ.|‿͜͡")
# Last resort when no native symbol exists (rare in the data).
_APPROX = {"ɹ": "ɹ", "ɡ": "ɡ", "g": "ɡ", "ə": "ə", "ʌ": "ʌ", "ɐ": "ɐ", "ɑ": "ɑː", "ɔ": "ɔ", "ʊ": "ʊ", "ɪ": "ɪ", "i": "i", "ɛ": "ɛ"}
_NASAL_BASE = {"ə": "ɐ", "ʌ": "ɐ", "ɐ": "ɐ", "ɪ": "i", "ʊ": "u", "ɑ": "ɑ"}

_inventory = None   # set by install(): the full SYMBOLS phone set


def _resolve(base, long, aspirated, nasal, inventory):
    """One IPA segment -> list of inventory symbols, preferring the native one."""
    L, H = ("ː" if long else ""), ("ʰ" if aspirated else "")
    base = "ɡ" if base == "g" else base
    candidates = []
    if nasal:
        for b in (base, _NASAL_BASE.get(base, base)):
            candidates += [b + N + L, b + N]
    candidates += [base + H + L, base + H]
    if not H and not L:
        candidates.append(_APPROX.get(base, base))
    if long:
        candidates += [_APPROX.get(base, base) + "ː", {"ɑ": "ɑː"}.get(base, "")]
    for c in candidates:
        if c in inventory:
            return [c]
    if aspirated:                                  # e.g. ɽʰ -> ɽ h
        return _resolve(base, long, False, nasal, inventory) + ["h"]
    return [base]                                  # unknown -> <unk>, reported by the annotator


def _ipa_word(word, inventory=None):
    inventory = inventory or _inventory
    units = []
    for char in unicodedata.normalize("NFD", word):
        if char in _IGNORED:
            continue
        if char in "ː:" and units:
            units[-1][1] = True
        elif char == "ʰ" and units:
            units[-1][2] = True
        elif char == N and units:
            units[-1][3] = True
        elif unicodedata.combining(char):
            continue
        else:
            units.append([char, False, False, False])
    phones = []
    for base, long, aspirated, nasal in units:
        phones += _resolve(base, long, aspirated, nasal, inventory)
    return phones


# ---- Roman Urdu -> IPA string -> _ipa_word --------------------------------------------------

_LEXICON = {   # Roman spelling -> IPA as the dataset writes it
    "hai": "hɛ", "hay": "hɛ", "he": "hɛ", "hain": "hẽ", "hein": "hẽ", "hen": "hẽ",
    "mein": "mẽ", "main": "mẽ", "men": "mẽ", "me": "mẽ", "nahi": "nahiː", "nahin": "nahĩ", "nahee": "nahiː",
    "aur": "ɔr", "or": "ɔr", "ka": "kaː", "ki": "ki", "ke": "keː", "ko": "koː", "se": "seː", "ne": "neː",
    "par": "par", "pe": "peː", "yeh": "jeː", "ye": "jeː", "woh": "ʋoː", "wo": "ʋoː", "to": "toː", "bhi": "bʰiː",
    "hi": "hiː", "is": "ɪs", "us": "ʊs", "in": "ɪn", "un": "ʊn", "kya": "kjaː", "kyun": "kjũ", "kyon": "kjũ",
    "kuch": "kʊcʰ", "kuchh": "kʊcʰ", "tha": "tʰaː", "thi": "tʰiː", "the": "tʰeː", "thay": "tʰeː", "ho": "hoː",
    "hoon": "hũ", "hun": "hũ", "hum": "ham", "tum": "tʊm", "aap": "aːp", "ap": "aːp", "ek": "eːk", "do": "doː",
    "jo": "ɟoː", "jab": "ɟab", "tak": "tak", "allah": "allaːh", "mera": "meːraː", "meri": "meːriː",
    "mere": "meːreː", "apna": "apnaː", "apni": "apniː", "apne": "apneː", "sab": "sab", "ab": "ab",
}
# Extra spellings added after training. Kept OUT of _rules_digest so trained
# checkpoints still load (infer.py checks the frontend fingerprint). Training
# uses the ipa: path, so this only affects Roman input. Add words here freely.
_LEXICON_EXTRA = {
    "chai": "caːeː", "chaye": "caːeː", "chae": "caːeː", "chaaye": "caːeː", "chaey": "caːeː",
}
_RULES = (   # longest first; output is IPA
    ("chh", "cʰ"), ("Th", "ʈʰ"), ("Dh", "ɖʰ"), ("Rh", "ɽʰ"),
    ("kh", "kʰ"), ("gh", "ɡʰ"), ("ch", "c"), ("sh", "ʃ"), ("th", "tʰ"), ("dh", "dʰ"), ("ph", "pʰ"),
    ("bh", "bʰ"), ("jh", "ɟʰ"), ("zh", "ʒ"), ("ck", "k"),
    ("aa", "aː"), ("ee", "iː"), ("ii", "iː"), ("oo", "uː"), ("ai", "ɛ"), ("ay", "eː"), ("ey", "eː"),
    ("ei", "ɛ"), ("au", "ɔ"), ("ou", "ɔ"), ("ao", "aːoː"),
    ("T", "ʈ"), ("D", "ɖ"), ("R", "ɽ"),
)
_SINGLE = {"a": "a", "e": "eː", "i": "ɪ", "o": "oː", "u": "ʊ", "g": "ɡ", "j": "ɟ", "y": "j", "w": "ʋ",
           "v": "ʋ", "c": "k", "x": "ks", "r": "r"}
_VOWEL_LETTERS = set("aeiou")


def roman_to_ipa(word):
    if word.lower() in _LEXICON_EXTRA and not any(c in "TDR" for c in word[1:]):
        return _LEXICON_EXTRA[word.lower()]
    if word.lower() in _LEXICON and not any(c in "TDR" for c in word):
        return _LEXICON[word.lower()]
    # Capital T/D/R inside a word mark retroflex. A capital first letter is just
    # sentence case, and ALL CAPS words are read as plain lowercase.
    if word.isupper() and len(word) > 1:
        word = word.lower()
    word = word[:1].lower() + word[1:]
    if word.lower() in _LEXICON and not any(c in "TDR" for c in word):
        return _LEXICON[word.lower()]
    word = "".join(c if c in "TDR" else c.lower() for c in word.replace("'", ""))
    out, i, last = [], 0, ""
    while i < len(word):
        for letters, ipa in _RULES:
            if word.startswith(letters, i):
                out.append(ipa)
                i += len(letters)
                break
        else:
            ch, final = word[i], i == len(word) - 1
            if ch == "a" and final and len(word) > 1:
                ipa = "aː"
            elif ch == "i" and final and len(word) > 1:
                ipa = "iː"
            else:
                ipa = _SINGLE.get(ch, ch if ch.isalpha() else "")
            if ipa and not (ch not in _VOWEL_LETTERS and ch == last):   # "achha": say a doubled consonant once
                out.append(ipa)
            last = ch
            i += 1
            continue
        last = ""
    return "".join(out)


_TOKEN = re.compile(r"[A-Za-z']+|[,.?!;:\"()—…]")


def pieces(text):
    if text.startswith(IPA_PREFIX):
        words = [w for w in (_ipa_word(w) for w in text[len(IPA_PREFIX):].split()) if w]
        return [(False, [[(p, 0) for p in w] for w in words])]
    from tiny_voca.text import normalize_text
    result, words = [], []
    for token in _TOKEN.findall(normalize_text(text).replace("-", " ")):
        if token[0].isalpha() or token[0] == "'":
            phones = _ipa_word(roman_to_ipa(token))
            if phones:
                words.append([(p, 0) for p in phones])
        else:
            if words:
                result.append((False, words))
                words = []
            result.append((True, token))
    if words:
        result.append((False, words))
    return result


def _rules_digest():
    blob = json.dumps([URDU_PHONES, _LEXICON, _RULES, _SINGLE, _APPROX, VERSION], ensure_ascii=False)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]


# ---- tiny-voca integration -----------------------------------------------------------------

def install():
    """Extend tiny_voca's symbol inventory with Urdu phones and swap in this front
    end. Must run BEFORE tiny_voca.{data,align,annotate,train,infer} are imported."""
    global _inventory
    import dataclasses
    import importlib
    import tiny_voca
    from tiny_voca import config as config_module, text

    if getattr(text, "_urdu_installed", False):
        return
    english = text.SYMBOLS
    assert len(english) == 76 and not set(URDU_PHONES) & set(english)
    text.CONSONANTS = text.CONSONANTS + URDU_CONSONANTS
    text.VOWELS = text.VOWELS + URDU_VOWELS
    text.PHONES = text.PHONES + URDU_PHONES
    text.SYMBOLS = english + URDU_PHONES          # English ids unchanged; Urdu appended
    text.SYMBOL_TO_ID = {s: i for i, s in enumerate(text.SYMBOLS)}
    text.PHONE_SET = frozenset(text.PHONES)
    text.PHONE_IDS = frozenset(text.SYMBOL_TO_ID[p] for p in text.PHONES) | {text.SYMBOL_TO_ID[text.UNK]}
    text.VOICELESS = text.VOICELESS | frozenset(URDU_VOICELESS)
    text.VOWEL_SET = frozenset(text.VOWELS)
    _inventory = text.PHONE_SET

    def phonemize(value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError("text must not be empty")
        return text.tokenize(pieces(value))

    def frontend_identity():
        return {"frontend_version": VERSION, "phonemizer": "tiny-urdu", "rules": _rules_digest(), "symbols": list(text.SYMBOLS)}

    text.phonemize = phonemize
    text.frontend_identity = frontend_identity
    text.frontend_runtime = lambda: {"phonemizer": "tiny-urdu-" + _rules_digest()}

    # Config: vocab_size default follows the larger inventory.
    UrduConfig = dataclasses.make_dataclass(
        "Config", [("vocab_size", int, dataclasses.field(default=len(text.SYMBOLS)))],
        bases=(config_module.Config,), frozen=True)
    UrduConfig.__module__ = config_module.__name__
    config_module.Config = UrduConfig
    tiny_voca.Config = UrduConfig
    # model.py was imported by tiny_voca/__init__ with the old symbol sets: re-import it.
    model = importlib.reload(importlib.import_module("tiny_voca.model"))
    tiny_voca.TinyVoca = model.TinyVoca
    text._urdu_installed = True

## Find the two models

In [ ]:
import glob, json, os
def find(tag):
    hits = [p for p in glob.glob("/kaggle/input/**/model.onnx", recursive=True) if f"/{tag}/" in p]
    assert hits, f"no {tag}/model.onnx under /kaggle/input: add the dataset as Input"
    return hits[0]
MODELS = {"1k": find("1k"), "5k": find("5k")}
for k, v in MODELS.items(): print(k, v, json.load(open(v.replace(".onnx", ".json")))["step"], "steps")

## Unseen test sentences

In [ ]:
import io, itertools, numpy as np, soundfile as sf, librosa
from datasets import load_dataset, Audio
ds = load_dataset("kaab4321/UrduTTS", split="test", streaming=True).cast_column("audio", Audio(decode=False))
rows = []
for r in itertools.islice(ds, SKIP, None):
    wav, sr = sf.read(io.BytesIO(r["audio"]["bytes"]))
    wav = wav.mean(1) if wav.ndim > 1 else wav
    secs = len(wav) / sr
    if 1.5 <= secs <= 12 and r["phonemes"].strip() and r["transcription"].strip():
        rows.append({"ipa": r["phonemes"], "ref": r["transcription"], "real": librosa.resample(wav.astype("float32"), orig_sr=sr, target_sr=16000)})
    if len(rows) == N: break
print(len(rows), "sentences")

## Each model speaks the sentences (on CPU, from the exported ONNX files)

In [ ]:
import sys, onnxruntime as ort
sys.path.insert(0, "urdu"); import frontend

def speak_all(onnx_path):
    meta = json.load(open(onnx_path.replace(".onnx", ".json")))
    sym = meta["symbols"]; ids = {s: i for i, s in enumerate(sym)}
    special = {"<pad>", "<bos>", "<eos>", "<sp>", "<unk>", ",", ".", "?", "!", ";", ":", '"', "'", "(", ")", "—", "…"}
    frontend._inventory = frozenset(s for s in sym if s not in special)
    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    voice = meta["voices"].index("M2")
    out = []
    for r in rows:
        words = frontend.pieces("ipa:" + r["ipa"])[0][1]
        toks = ["<bos>"]
        for w in words:
            if toks[-1] not in ("<bos>", "<sp>"): toks.append("<sp>")
            toks += [p for p, _ in w]
        toks.append("<eos>")
        t = [ids.get(s, ids["<unk>"]) for s in toks]
        if len(t) > meta["max_tokens"] - 12: out.append(None); continue
        arr = lambda v: np.array([v], dtype=np.int64)
        audio = sess.run(None, {"tokens": arr(t), "stress": np.zeros((1, len(t)), np.int64), "voice": np.array([voice], np.int64),
                                "speed": np.array([1.0], np.float32), "pitch_shift": np.array([0.0], np.float32),
                                "pitch_range": np.array([1.0], np.float32)})[0][0]
        peak = np.abs(audio).max()
        if peak > 0.99: audio = audio * 0.99 / peak
        out.append(librosa.resample(audio.astype("float32"), orig_sr=meta["sample_rate"], target_sr=16000))
    return out
AUDIO = {k: speak_all(v) for k, v in MODELS.items()}
print({k: sum(a is not None for a in v) for k, v in AUDIO.items()}, "spoken")

## Whisper listens and writes it down

In [ ]:
import torch
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model=WHISPER, torch_dtype=torch.float16, device=0)
gen = {"language": "urdu", "task": "transcribe"}
def transcribe(clips):
    res = asr([{"raw": c, "sampling_rate": 16000} for c in clips], batch_size=8, generate_kwargs=gen)
    return [x["text"] for x in res]
keep = [i for i in range(len(rows)) if all(AUDIO[k][i] is not None for k in AUDIO)]
HEARD = {"real recordings": transcribe([rows[i]["real"] for i in keep])}
for k in AUDIO: HEARD[f"{k} model"] = transcribe([AUDIO[k][i] for i in keep])
REFS = [rows[i]["ref"] for i in keep]
print(len(keep), "sentences scored")

## Word error rate

In [ ]:
import re, jiwer
def norm(s):
    s = re.sub(r"[\u064B-\u065F\u0670\u0640]", "", s)                       # diacritics, tatweel
    s = s.translate(str.maketrans({"ي": "ی", "ى": "ی", "ك": "ک", "ۀ": "ہ", "ة": "ہ", "ه": "ہ", "ۃ": "ہ"}))
    s = re.sub(r"[^\w\s]|_", " ", s)                                          # punctuation (including ، ۔ ؟)
    return re.sub(r"\s+", " ", s).strip()
refs = [norm(r) for r in REFS]
print(f"{'':18} {'WER %':>8} {'CER %':>8}")
for name, heard in HEARD.items():
    h = [norm(x) for x in heard]
    print(f"{name:18} {100*jiwer.wer(refs, h):8.1f} {100*jiwer.cer(refs, h):8.1f}")
# sentence by sentence: which model is closer?
e = {n: [jiwer.wer(r, norm(x)) for r, x in zip(refs, h)] for n, h in HEARD.items()}
better = sum(a < b for a, b in zip(e["5k model"], e["1k model"])); worse = sum(a > b for a, b in zip(e["5k model"], e["1k model"]))
print(f"\n5k better on {better} sentences, worse on {worse}, same on {len(refs)-better-worse}")

## Listen to a few (1k vs 5k)

In [ ]:
import IPython.display as ipd
for j in range(5):
    i = keep[j]; print(REFS[j])
    for k in AUDIO:
        print(" ", k, "->", HEARD[f"{k} model"][j]); ipd.display(ipd.Audio(AUDIO[k][i], rate=16000))